# Unit 8: Climate & Energy - Embedding Pipeline Illustration for Catalyst Screening

**Companion notebook for *What Quantum Computers Are Actually For***

**Notebook class:** pipeline illustration


**Code note:** This notebook is written as teaching code rather than library code. The Quokka calls, circuit construction, and measurement post-processing stay explicit on purpose so you can inspect the mechanism end to end. A production library would factor more of this behind helpers; here we keep the moving parts visible.

This notebook keeps the capstone honest by illustrating one embedded active-space solve step inside a catalyst-screening workflow.

It does **not** run DFT, DMET bath construction, or a full self-consistent embedding loop. Instead, it starts from a precomputed 2-qubit embedded Hamiltonian for a toy catalyst active site, runs a direct-measurement VQE solve on Quokka, and shows where that solve step sits inside the larger pipeline.

**What you'll see:**
1. A precomputed embedded active-space Hamiltonian for a catalyst toy model
2. A classical embedding baseline and an exact benchmark for that reduced model
3. One direct-measurement VQE solve step on Quokka
4. A map of which parts of the full embedding workflow are executed here and which are only represented structurally

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

import requests
from requests.packages.urllib3.exceptions import InsecureRequestWarning
requests.packages.urllib3.disable_warnings(InsecureRequestWarning)

QUOKKA = "quokka1"
QUOKKA_URL = f"http://{QUOKKA}.quokkacomputing.com/qsim/qasm"


def _decode_quokka_counts(payload: dict) -> dict:
    """Normalize Quokka responses to a simple {bitstring: count} mapping."""
    if isinstance(payload, dict) and all(isinstance(v, int) for v in payload.values()):
        return payload

    if not isinstance(payload, dict):
        raise TypeError(f"Unexpected Quokka payload type: {type(payload)!r}")

    if payload.get("error_code", 0) not in (0, None):
        raise RuntimeError(f"Quokka error {payload.get('error_code')}: {payload.get('error')}")

    result = payload.get("result")
    if not isinstance(result, dict) or "c" not in result:
        raise ValueError(f"Unexpected Quokka result format: {payload}")

    counts = {}
    for shot in result["c"]:
        bitstring = ''.join(str(int(bit)) for bit in shot)
        counts[bitstring] = counts.get(bitstring, 0) + 1
    return counts


def run_qasm(program: str, shots: int = 1024) -> dict:
    result = requests.post(QUOKKA_URL, json={"script": program, "count": shots}, verify=False)
    result.raise_for_status()
    return _decode_quokka_counts(result.json())


print(f"Connected to {QUOKKA_URL}")

## 1. A precomputed embedded active-space toy model

Think of this as the output of the classical part of an embedding workflow:

- a classical calculation has already selected an active site,
- the environment has already been compressed into an effective potential, and
- the result is a small qubit Hamiltonian for the embedded fragment.

The notebook starts **after** those classical steps. That is why this is a pipeline illustration rather than a faithful catalyst calculation.

In [ ]:
def embedded_active_space_coeffs() -> dict:
    """Return a precomputed 2-qubit embedded Hamiltonian for a catalyst toy model."""
    return {
        'II': -151.84392077,
        'Z0': +0.4215,
        'Z1': -0.2876,
        'Z0Z1': +0.3142,
        'X0X1': +0.1253,
        'Y0Y1': +0.1253,
    }


def exact_diagonalisation_energy(coeffs: dict) -> float:
    I = np.eye(2)
    X = np.array([[0, 1], [1, 0]], dtype=float)
    Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
    Z = np.array([[1, 0], [0, -1]], dtype=float)
    hamiltonian = (
        coeffs['II'] * np.kron(I, I)
        + coeffs['Z0'] * np.kron(I, Z)
        + coeffs['Z1'] * np.kron(Z, I)
        + coeffs['Z0Z1'] * np.kron(Z, Z)
        + coeffs['X0X1'] * np.kron(X, X)
        + coeffs['Y0Y1'] * np.kron(Y, Y)
    )
    return float(np.min(np.linalg.eigvalsh(hamiltonian)))


active_space_coeffs = embedded_active_space_coeffs()
E_dft = -152.8200
E_exact_active = exact_diagonalisation_energy(active_space_coeffs)

print('Catalyst toy model: embedded active site for CO binding')
print(f'  Classical embedding baseline: {E_dft:.4f} Ha')
print(f'  Exact embedded benchmark:     {E_exact_active:.4f} Ha')
print(f'  Embedded correction:          {E_exact_active - E_dft:.4f} Ha')
print(f'                                {(E_exact_active - E_dft) * 627.5:.1f} kcal/mol')

## 2. One embedded VQE solve step on Quokka

This is the only genuinely quantum step in the notebook.

The ansatz and measurement pattern are intentionally the same kind of reduced-model move used earlier in Unit 3: a small precomputed Hamiltonian, a one-parameter circuit, and direct measurements of the Pauli terms. What changes is the interpretation. Here, the solve step is standing in for the quantum subroutine inside a larger embedding loop.

In [ ]:
def vqe_measurement_circuit(theta: float, basis: str = 'Z') -> str:
    basis_rotation_lines = []
    if basis == 'X':
        basis_rotation_lines = ['h q[0];', 'h q[1];']
    elif basis == 'Y':
        basis_rotation_lines = ['sdg q[0];', 'h q[0];', 'sdg q[1];', 'h q[1];']

    basis_rotations = '' if not basis_rotation_lines else '\n'.join(basis_rotation_lines) + '\n'

    return f"""OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];

// Reference state for the reduced embedded model
x q[1];

// One-parameter entangling ansatz
ry({theta:.6f}) q[0];
cx q[0], q[1];

{basis_rotations}measure q[0] -> c[0];
measure q[1] -> c[1];
"""


def expectation_from_counts(counts: dict, observable: str) -> float:
    total = sum(counts.values())
    expectation = 0.0
    for bitstring, count in counts.items():
        b0, b1 = int(bitstring[0]), int(bitstring[1])
        z0 = 1 - 2 * b0
        z1 = 1 - 2 * b1
        if observable == 'Z0':
            eigenvalue = z0
        elif observable == 'Z1':
            eigenvalue = z1
        else:
            eigenvalue = z0 * z1
        expectation += eigenvalue * count / total
    return expectation


def compute_active_energy(theta: float, coeffs: dict, shots: int = 1024) -> float:
    z_counts = run_qasm(vqe_measurement_circuit(theta, basis='Z'), shots=shots)
    xx_counts = run_qasm(vqe_measurement_circuit(theta, basis='X'), shots=shots)
    yy_counts = run_qasm(vqe_measurement_circuit(theta, basis='Y'), shots=shots)

    energy = coeffs['II']
    energy += coeffs['Z0'] * expectation_from_counts(z_counts, 'Z0')
    energy += coeffs['Z1'] * expectation_from_counts(z_counts, 'Z1')
    energy += coeffs['Z0Z1'] * expectation_from_counts(z_counts, 'Z0Z1')
    energy += coeffs['X0X1'] * expectation_from_counts(xx_counts, 'XX')
    energy += coeffs['Y0Y1'] * expectation_from_counts(yy_counts, 'YY')
    return float(energy)


thetas = np.linspace(0, np.pi, 13)
energies = [compute_active_energy(theta, active_space_coeffs, shots=1024) for theta in thetas]
best_idx = int(np.argmin(energies))
theta_opt = float(thetas[best_idx])
E_vqe = float(energies[best_idx])
solver_gap_mha = abs(E_vqe - E_exact_active) * 1000

print(f'Embedded VQE optimum: theta = {theta_opt:.3f}, E = {E_vqe:.4f} Ha')
print(f'Exact embedded benchmark:                {E_exact_active:.4f} Ha')
print(f'Classical embedding baseline:            {E_dft:.4f} Ha')
print(f'Gap to exact benchmark:                  {solver_gap_mha:.1f} mHa')
print('Interpretation: the notebook executes the quantum solve step, while active-space selection and embedding remain precomputed inputs.')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(thetas, energies, 'o-', color='#009688', label='Embedded VQE on Quokka')
ax1.axhline(y=E_exact_active, color='red', linestyle='--', label=f'Exact embedded benchmark = {E_exact_active:.4f} Ha')
ax1.axhline(y=E_dft, color='orange', linestyle=':', label=f'Classical embedding baseline = {E_dft:.4f} Ha')
ax1.set_xlabel('theta')
ax1.set_ylabel('Energy (Ha)')
ax1.set_title('Embedded active-space solve step')
ax1.legend(fontsize=9)

methods = ['Classical
baseline', 'Embedded
VQE', 'Exact
benchmark']
values = [E_dft, E_vqe, E_exact_active]
colors = ['#FF9800', '#009688', '#F44336']
ax2.bar(methods, values, color=colors, edgecolor='k', linewidth=0.5)
ax2.set_ylabel('Energy (Ha)')
ax2.set_title('Toy embedded model comparison')
ax2.set_ylim(min(values) - 0.08, max(values) + 0.08)

plt.tight_layout()
plt.show()

## 3. What the real embedding pipeline would add

| Stage | Real workflow | Notebook treatment |
|------|------|------|
| Select active space | choose strongly correlated orbitals from a classical calculation | represented as a preselected 2-qubit toy active space |
| Embed the environment | build an effective Hamiltonian with DFT, DMET, or a related method | represented by precomputed coefficients |
| Solve the embedded problem | run VQE or QPE on the embedded Hamiltonian | executed here on Quokka |
| Update self-consistently | feed the embedded result back into the classical environment and iterate | omitted from the runnable notebook |
| Interpret chemistry | turn energies into binding or activation trends | discussed conceptually, not computed here |

That is the honest role of this notebook: it demonstrates where the quantum subroutine lives inside the larger catalyst-screening workflow.

## What's next?

- [Deep-Dive 8 - Quantum Embedding Methods](https://github.com/johnazariah/quantum-bottleneck/blob/main/manuscript/16-quantum-embedding.md): the full select -> embed -> solve -> iterate story behind this illustration
- Replace the precomputed coefficients with a real classical embedding front end if you want a faithful catalyst workflow
- [Unit 3 - Drug Discovery](https://github.com/johnazariah/quantum-bottleneck/blob/main/manuscript/05-drug-discovery.md): the earlier reduced-model VQE anatomy that this capstone build reuses